In [9]:

import numpy as np
import pandas as pd


from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold

from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve, auc, precision_score, recall_score
from tqdm.auto import tqdm

import pickle


In [10]:
df=pd.read_csv('https://raw.githubusercontent.com/alexeygrigorev/mlbookcamp-code/master/chapter-03-churn-prediction/WA_Fn-UseC_-Telco-Customer-Churn.csv')

In [11]:
df.columns=df.columns.str.lower().str.replace(' ', '_')
categorical_variables=list((df.dtypes[df.dtypes=='str'].index).drop('totalcharges'))

for c in categorical_variables:
    df[c]=df[c].str.lower().str.replace(' ', '_')
    df[c]=df[c].fillna('NA')

categorical_variables.remove('churn')
df['totalcharges']=pd.to_numeric(df['totalcharges'].str.strip().str.replace(' ', ''), errors='coerce')
df.totalcharges = df.totalcharges.fillna(0)
numerical_variable=list(df.dtypes[((df.dtypes=='int64') | (df.dtypes=='float64'))].index)
df.churn=(df.churn=='yes').astype('int')

In [21]:
df.head(1).to_dict(orient='records')

[{'customerid': '7590-vhveg',
  'gender': 'female',
  'seniorcitizen': 0,
  'partner': 'yes',
  'dependents': 'no',
  'tenure': 1,
  'phoneservice': 'no',
  'multiplelines': 'no_phone_service',
  'internetservice': 'dsl',
  'onlinesecurity': 'no',
  'onlinebackup': 'yes',
  'deviceprotection': 'no',
  'techsupport': 'no',
  'streamingtv': 'no',
  'streamingmovies': 'no',
  'contract': 'month-to-month',
  'paperlessbilling': 'yes',
  'paymentmethod': 'electronic_check',
  'monthlycharges': 29.85,
  'totalcharges': 29.85,
  'churn': 0}]

In [6]:
df_full_train,df_test=train_test_split(
    df,
    test_size=0.2,
    random_state=1
)
df_train,df_val=train_test_split(
    df_full_train,
    test_size=0.25,
    random_state=1
)

df_full_train=df_full_train.reset_index(drop=True)
df_train=df_train.reset_index(drop=True)
df_val=df_val.reset_index(drop=True)
df_test=df_test.reset_index(drop=True)

y_full_train=df_full_train.churn.values
y_train=df_train.churn.values
y_val=df_val.churn.values
y_test=df_test.churn.values


In [ ]:
def train(df,y_train, C=1.0):
    
    dicts=df[categorical_variables+numerical_variable].to_dict(orient="records")
    dv=DictVectorizer(sparse=False)
    
    X=dv.fit_transform(dicts)
    model=LogisticRegression(C=C, max_iter= 1000)
    model.fit(X,y_train)
    
    return dv, model

In [8]:
def predict(df, dv, model):
    
    dicts=df[categorical_variables+numerical_variable].to_dict(orient="records")
    X=dv.transform(dicts)
    
    y_pred=model.predict_proba(X)[:,1]
    
    return y_pred

In [ ]:
def cross_validation(df, number_of_splits, shuffle, random_state , C=1.0):
    
    score=[]
    kfold=KFold(n_splits=number_of_splits,
                shuffle=shuffle,
                random_state=random_state
                )
    for train_idx, val_idx in tqdm(kfold.split(df),total=number_of_splits):
        
        df_train=df.iloc[train_idx]
        df_val=df.iloc[val_idx]
        
        y_train=df_train.churn.values
        y_val=df_val.churn.values
        
        
        dv, model=train(df_train, y_train, c)
        y_pred= predict(df_val, dv, model)
        
        auc=roc_auc_score(y_val, y_pred)
        score.append(auc)
    
    return score        

In [ ]:
cros_score={}
for C in [0.0001, 0.01]:
    score=cross_validation(df=df_full_train,number_of_splits=5, shuffle=True, random_state= 1 , C=C)
    cros_score[c]= {'mean': float(round(np.mean(score),3)), 'std': float(round(np.std(score),3))}
    

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/5 [00:00<?, ?it/s]

d:\ML Zoomcamp\ML_zoo_2026\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
d:\ML Zoomcamp\ML_zoo_2026\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing

In [11]:
best_C = None
best_mean = -1
best_std = float("inf")

for C, r in cros_score.items():
    mean = round(r["mean"], 3)
    std = round(r["std"], 3)

    if mean > best_mean or (mean == best_mean and std < best_std):
        best_C = C
        best_mean = mean
        best_std = std

dv, model=train(df_full_train, y_full_train, best_C)
y_pred= predict(df_test,dv,model)

auc=roc_auc_score(y_test,y_pred)
auc


d:\ML Zoomcamp\ML_zoo_2026\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


0.8548850574712645

Saving the Model

In [12]:
output_file = f'model_c={best_C}.bin'
# f_out= open(output_file, 'wb')
# pickle.dump((dv,model), f_out)
# f_out.close()

In [13]:
with open(output_file, 'wb') as f_out:
    pickle.dump((dv,model), f_out)
    

Loading the model

In [5]:
import pickle

In [3]:
model_file = f'model_c=0.01.bin'

In [6]:
with open(model_file, 'rb') as f_in:
    dv, model = pickle.load(f_in)

In [7]:
dv, model

(DictVectorizer(sparse=False), LogisticRegression(C=0.01, max_iter=1000))